# Build a GPT-style LLM From Scratch — Complete Walkthrough

This notebook walks through **every stage** of building a GPT-style Large Language Model
from scratch using the modular [`src/`](../src) package, then:

1. **Pretrains** a small model on a text corpus and generates text.
2. **Loads OpenAI's pretrained GPT-2 weights** into the same architecture and generates
   much higher-quality text.

**Pipeline overview**

```
Text → Tokenizer → Dataset (sliding window) → GPTModel → logits
                                                   ↓
                                      cross-entropy loss → AdamW → repeat
                                                   ↓
                                      generate() → new text
```

Every component (tokenizer, attention, transformer block, training loop, generation)
lives in `src/` and is written with only PyTorch primitives + NumPy — no high-level ML
frameworks for the core model logic.

> **Run order:** execute the cells top-to-bottom. The pretraining section trains a tiny
> model in a minute or two on CPU. The final section (loading GPT-2 weights) requires
> `tensorflow` + `tqdm` and downloads ~500 MB the first time.

## 0. Setup

Make the repository root importable so `import src` works from the notebook, then import
the pieces we need. We also set a manual seed for reproducibility.

In [ ]:
import sys
from pathlib import Path

# Add the repo root (parent of this notebook's folder) to the import path.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "src").exists():
    pass
elif (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)

import torch
import tiktoken

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)

## 1. Tokenization

Language models operate on **token IDs**, not raw text. The first step is converting text
into integers and back.

### 1a. Rule-based tokenizer (from scratch)

`SimpleTokenizerV1` splits text on whitespace/punctuation with a regex and maps each unique
token to an integer. `SimpleTokenizerV2` adds an `<|unk|>` token so it can handle words it
never saw during vocabulary construction.

In [ ]:
from src.tokenizer import SimpleTokenizerV1, SimpleTokenizerV2

sample = "It's the last he painted, you know, Mrs. Gisburn said with pardonable pride."

tok_v1 = SimpleTokenizerV1(sample)          # builds vocab from this text
ids = tok_v1.encode(sample)
print("Vocab size:", len(tok_v1.str_to_int))
print("Encoded:", ids[:12], "...")
print("Decoded:", tok_v1.decode(ids))

In [ ]:
# V2 gracefully handles out-of-vocabulary words with <|unk|>
tok_v2 = SimpleTokenizerV2(sample)
print(tok_v2.encode("It's a brandnewword here"))
print(tok_v2.decode(tok_v2.encode("It's a brandnewword here")))

### 1b. Byte Pair Encoding (BPE)

Rule-based tokenizers can't handle arbitrary text. Real GPT models use **Byte Pair
Encoding**, which breaks unknown words into sub-word pieces so *any* string can be encoded
with a fixed 50,257-token vocabulary. `BPETokenizer` wraps OpenAI's `tiktoken`.

In [ ]:
from src.tokenizer import BPETokenizer

bpe = BPETokenizer()
print("BPE vocab size:", bpe.vocab_size)

ids = bpe.encode("Akwirw ier — a made-up word the model has never seen.")
print("Token IDs:", ids)
print("Round-trip:", bpe.decode(ids))

# Every ID maps back to a sub-word piece:
for i in ids[:8]:
    print(f"  {i:>6} -> {bpe.decode([i])!r}")

## 2. The Data Pipeline

For next-token prediction, each training example is a sequence of tokens (the **input**)
paired with the same sequence shifted one position to the right (the **target**).

`GPTDataset` slides a window of length `max_length` across the tokenized corpus with a
given `stride`, and `create_dataloader` batches these pairs.

In [ ]:
from src.data import create_dataloader

text = (REPO_ROOT / "data" / "the-verdict.txt").read_text(encoding="utf-8")
print("Corpus characters:", len(text))

loader = create_dataloader(
    text, tokenizer=bpe, batch_size=4, max_length=8, stride=8, shuffle=False,
)
inputs, targets = next(iter(loader))
print("Input batch shape:", inputs.shape)     # (batch, max_length)
print("Target batch shape:", targets.shape)

print("\nFirst example (input vs target shifted by 1):")
print("input :", inputs[0].tolist())
print("target:", targets[0].tolist())
print("input  text:", repr(bpe.decode(inputs[0].tolist())))
print("target text:", repr(bpe.decode(targets[0].tolist())))

## 3. Embeddings

Token IDs are turned into dense vectors by an **embedding layer**, and a separate
**positional embedding** encodes each token's position (attention itself is
order-agnostic). Inside `GPTModel` these two are simply added together. Here's the idea in
isolation:

In [ ]:
import torch.nn as nn

vocab_size, emb_dim, context_len = bpe.vocab_size, 16, 8
tok_emb = nn.Embedding(vocab_size, emb_dim)
pos_emb = nn.Embedding(context_len, emb_dim)

token_embeddings = tok_emb(inputs)                                  # (batch, seq, emb_dim)
position_embeddings = pos_emb(torch.arange(inputs.shape[1]))        # (seq, emb_dim)
input_embeddings = token_embeddings + position_embeddings           # broadcast add
print("token embeddings :", token_embeddings.shape)
print("position embeddings:", position_embeddings.shape)
print("input embeddings :", input_embeddings.shape)

## 4. Attention Mechanism

The heart of a transformer is **multi-head causal self-attention**. Each token computes
query/key/value vectors, attends to *previous* tokens only (the causal mask), and produces
a context-aware representation. `MultiHeadAttention` runs several such heads in parallel.

In [ ]:
from src.model import MultiHeadAttention

torch.manual_seed(123)
batch_example = torch.rand(2, 6, 16)   # (batch, num_tokens, d_in)

mha = MultiHeadAttention(
    d_in=16, d_out=16, context_length=6, dropout=0.0, num_heads=2, qkv_bias=False,
)
context_vecs = mha(batch_example)
print("Input shape :", batch_example.shape)
print("Output shape:", context_vecs.shape)   # same shape, now context-aware

## 5. The Full GPT Architecture

A `GPTModel` stacks:

- **Token + positional embeddings**
- `n_layers` × **`TransformerBlock`** — each is *pre-norm* multi-head attention +
  feed-forward, both wrapped in residual connections
- A final **LayerNorm**
- An **output head** projecting back to vocabulary logits

We use a *small* configuration here so it trains quickly on CPU. `get_config` starts from
the GPT-2 preset and lets us override any hyperparameter.

In [ ]:
from src.config import get_config
from src.model import GPTModel

GPT_CONFIG = get_config(
    "gpt2-small (124M)",
    context_length=256,   # shorter context = faster demo
    emb_dim=256,          # smaller model
    n_heads=8,
    n_layers=6,
    drop_rate=0.1,
    qkv_bias=False,
)
GPT_CONFIG

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG)
print(f"Total parameters: {model.num_params():,}")

# A forward pass maps token IDs -> logits over the vocabulary.
logits = model(inputs)
print("logits shape:", logits.shape)   # (batch, seq_len, vocab_size)

## 6. Text Generation (untrained model)

`generate_text_simple` does greedy decoding: repeatedly feed the context, take the most
likely next token, append it, repeat. With **untrained** weights the output is gibberish —
which is exactly what we expect before training.

In [ ]:
from src.generation import generate_text_simple, text_to_token_ids, token_ids_to_text

start_context = "Every effort moves you"
model.eval()
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, bpe),
    max_new_tokens=15,
    context_size=GPT_CONFIG["context_length"],
)
print("Untrained output:\n", token_ids_to_text(token_ids, bpe))

## 7. Pretraining

Now we actually train the model to predict the next token. The **loss** is cross-entropy
between the predicted logits and the true next tokens.

### 7a. Loss and train/validation split

In [ ]:
from src.training import calc_loss_loader

# Split the raw text, then build a loader for each half.
train_frac = 0.90
split_idx = int(train_frac * len(text))
train_text, val_text = text[:split_idx], text[split_idx:]

train_loader = create_dataloader(
    train_text, tokenizer=bpe, batch_size=2,
    max_length=GPT_CONFIG["context_length"], stride=GPT_CONFIG["context_length"],
    shuffle=True, drop_last=True,
)
val_loader = create_dataloader(
    val_text, tokenizer=bpe, batch_size=2,
    max_length=GPT_CONFIG["context_length"], stride=GPT_CONFIG["context_length"],
    shuffle=False, drop_last=False,
)
print("train batches:", len(train_loader), "| val batches:", len(val_loader))

model.to(device)
with torch.no_grad():
    print("Initial train loss:", round(calc_loss_loader(train_loader, model, device), 3))
    print("Initial   val loss:", round(calc_loss_loader(val_loader, model, device), 3))

### 7b. The training loop

`train_model_simple` runs the standard loop: forward → loss → backprop → optimizer step,
logging losses every `eval_freq` steps and printing a sample generation each epoch.

> This trains on a tiny corpus, so the model will **overfit** and start reciting the text —
> that's fine, it demonstrates that learning is happening. Real pretraining uses billions of
> tokens.

In [ ]:
import time
from src.training import train_model_simple

torch.manual_seed(123)
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-4, weight_decay=0.1)

start = time.time()
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=10, eval_freq=5, eval_iter=5,
    start_context="Every effort moves you", tokenizer=bpe,
)
print(f"\nTraining finished in {(time.time() - start) / 60:.2f} min")

### 7c. Loss curve

In [ ]:
from src.utils import plot_losses

epochs_tensor = torch.linspace(0, 10, len(train_losses))
plot_losses(epochs_tensor, tokens_seen, train_losses, val_losses)

### 7d. Generate after training

Now with **temperature** and **top-k** sampling for more varied output. Compare this to the
gibberish from the untrained model in section 6.

In [ ]:
from src.generation import generate

model.eval()
token_ids = generate(
    model=model,
    idx=text_to_token_ids("Every effort moves you", bpe).to(device),
    max_new_tokens=25,
    context_size=GPT_CONFIG["context_length"],
    top_k=25,
    temperature=1.0,
)
print("Trained output:\n", token_ids_to_text(token_ids, bpe))

### 7e. Save the checkpoint

In [ ]:
from src.utils import save_checkpoint

save_checkpoint("model.pth", model, optimizer)
print("Saved to model.pth")

## 8. Loading Pretrained GPT-2 Weights

Training a good LLM from scratch takes enormous compute. Instead, we can load **OpenAI's
pretrained GPT-2 weights** into our *identical* architecture and immediately get
high-quality generation. This proves our from-scratch implementation is weight-compatible
with the real thing.

> **Requirements:** `pip install tensorflow tqdm`. The first run downloads ~500 MB into a
> local `gpt2/` folder. Skip this section if you only care about the from-scratch build.

In [ ]:
# Uncomment to install the optional dependencies for this section:
# %pip install tensorflow tqdm

In [ ]:
from src.utils.gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")
print("Settings:", settings)
print("Param keys:", list(params.keys()))
print("Token embedding shape:", params["wte"].shape)

### 8a. Build a matching model and load the weights

GPT-2 uses the **full** 124M config: 768-dim embeddings, 12 heads, 12 layers, a 1024-token
context, and **bias terms** in the QKV projections. We build that exact config, then copy
the pretrained weights in with `load_weights_into_gpt`.

In [ ]:
from src.utils import load_weights_into_gpt

gpt2_config = get_config("gpt2-small (124M)", context_length=1024, qkv_bias=True)
gpt2 = GPTModel(gpt2_config)
gpt2.eval()

load_weights_into_gpt(gpt2, params)
gpt2.to(device)
print(f"Loaded GPT-2 124M — {gpt2.num_params():,} parameters")

### 8b. Generate with real GPT-2 weights

The same `generate` function, but now backed by pretrained weights — the output is
coherent English.

In [ ]:
torch.manual_seed(123)
token_ids = generate(
    model=gpt2,
    idx=text_to_token_ids("Every effort moves you", bpe).to(device),
    max_new_tokens=40,
    context_size=gpt2_config["context_length"],
    top_k=50,
    temperature=1.0,
)
print(token_ids_to_text(token_ids, bpe))

In [ ]:
# Try your own prompt:
prompt = "The meaning of life is"
out = generate(
    model=gpt2,
    idx=text_to_token_ids(prompt, bpe).to(device),
    max_new_tokens=50,
    context_size=gpt2_config["context_length"],
    top_k=50,
    temperature=0.8,
)
print(token_ids_to_text(out, bpe))

## 9. Summary

We built and used a complete GPT-style LLM from scratch:

| Stage | Component | `src` module |
|-------|-----------|--------------|
| 1 | Tokenization (rule-based + BPE) | `src.tokenizer` |
| 2 | Sliding-window data pipeline | `src.data` |
| 3 | Embeddings | (inside `src.model.gpt`) |
| 4 | Multi-head causal attention | `src.model.attention` |
| 5 | Transformer blocks + full GPT | `src.model` |
| 6–7 | Loss, pretraining loop, generation | `src.training`, `src.generation` |
| 8 | Loading pretrained GPT-2 weights | `src.utils.gpt_download`, `src.utils.gpt2_weights` |

**Next steps to explore:**
- Train on a larger corpus (e.g. the Harry Potter books) for many more epochs.
- Load the larger GPT-2 sizes (`355M`, `774M`, `1558M`) via `get_config` + `download_and_load_gpt2`.
- Fine-tune the pretrained model on your own domain data.